# Exploratory analysis with SQL

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## SQLite dataset
This independently supplied SQL dataset has 101 rows and four launch-site spellings. It is not the 90-row modeling dataset.

In [2]:
import sqlite3
con=sqlite3.connect(':memory:')
sql_df=pd.read_csv(DATA/'Spacex.csv')
sql_df.to_sql('SPACEXTBL',con,index=False,if_exists='replace')
print(sql_df.shape)
answers={}


(101, 10)


## Task 1: Unique launch sites

In [3]:
query='SELECT DISTINCT Launch_Site FROM SPACEXTBL ORDER BY Launch_Site'
print(query)
result=pd.read_sql_query(query,con)
answers['1']=result.to_dict('records')
display(result)

SELECT DISTINCT Launch_Site FROM SPACEXTBL ORDER BY Launch_Site


,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


## Task 2: First five CCA launches

In [4]:
query="SELECT Date, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' ORDER BY Date LIMIT 5"
print(query)
result=pd.read_sql_query(query,con)
answers['2']=result.to_dict('records')
display(result)

SELECT Date, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' ORDER BY Date LIMIT 5


,Date,Booster_Version,Launch_Site
0,2010-06-04,F9 v1.0 B0003,CCAFS LC-40
1,2010-12-08,F9 v1.0 B0004,CCAFS LC-40
2,2012-05-22,F9 v1.0 B0005,CCAFS LC-40
3,2012-10-08,F9 v1.0 B0006,CCAFS LC-40
4,2013-03-01,F9 v1.0 B0007,CCAFS LC-40


## Task 3: NASA CRS total payload

In [5]:
query="SELECT SUM(PAYLOAD_MASS__KG_) AS Total_kg FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'"
print(query)
result=pd.read_sql_query(query,con)
answers['3']=result.to_dict('records')
display(result)

SELECT SUM(PAYLOAD_MASS__KG_) AS Total_kg FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'


,Total_kg
0,45596


## Task 4: F9 v1.1 average payload

In [6]:
query="SELECT AVG(PAYLOAD_MASS__KG_) AS Average_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'"
print(query)
result=pd.read_sql_query(query,con)
answers['4']=result.to_dict('records')
display(result)

SELECT AVG(PAYLOAD_MASS__KG_) AS Average_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'


,Average_kg
0,2534.666667


## Task 5: First successful ground landing

In [7]:
query="SELECT MIN(Date) AS First_date FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'"
print(query)
result=pd.read_sql_query(query,con)
answers['5']=result.to_dict('records')
display(result)

SELECT MIN(Date) AS First_date FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'


,First_date
0,2015-12-22


## Task 6: Drone success, payload 4000 to 6000 kg

In [8]:
query="SELECT Date,Booster_Version,PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_>4000 AND PAYLOAD_MASS__KG_<6000 ORDER BY Date"
print(query)
result=pd.read_sql_query(query,con)
answers['6']=result.to_dict('records')
display(result)

SELECT Date,Booster_Version,PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_>4000 AND PAYLOAD_MASS__KG_<6000 ORDER BY Date


,Date,Booster_Version,PAYLOAD_MASS__KG_
0,2016-05-06,F9 FT B1022,4696
1,2016-08-14,F9 FT B1026,4600
2,2017-03-30,F9 FT B1021.2,5300
3,2017-10-11,F9 FT B1031.2,5200


## Task 7: Mission outcomes

In [9]:
query='SELECT TRIM(Mission_Outcome) AS Mission_Outcome,COUNT(*) AS Missions FROM SPACEXTBL GROUP BY TRIM(Mission_Outcome) ORDER BY Missions DESC'
print(query)
result=pd.read_sql_query(query,con)
answers['7']=result.to_dict('records')
display(result)

SELECT TRIM(Mission_Outcome) AS Mission_Outcome,COUNT(*) AS Missions FROM SPACEXTBL GROUP BY TRIM(Mission_Outcome) ORDER BY Missions DESC


,Mission_Outcome,Missions
0,Success,99
1,Success (payload status unclear),1
2,Failure (in flight),1


## Task 8: Boosters carrying maximum payload

In [10]:
query='SELECT Date,Booster_Version,PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL) ORDER BY Date'
print(query)
result=pd.read_sql_query(query,con)
answers['8']=result.to_dict('records')
display(result)

SELECT Date,Booster_Version,PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL) ORDER BY Date


,Date,Booster_Version,PAYLOAD_MASS__KG_
0,2019-11-11,F9 B5 B1048.4,15600
1,2020-01-07,F9 B5 B1049.4,15600
2,2020-01-29,F9 B5 B1051.3,15600
3,2020-02-17,F9 B5 B1056.4,15600
4,2020-03-18,F9 B5 B1048.5,15600
5,2020-04-22,F9 B5 B1051.4,15600
6,2020-06-04,F9 B5 B1049.5,15600
7,2020-09-03,F9 B5 B1060.2,15600
8,2020-10-06,F9 B5 B1058.3,15600
9,2020-10-18,F9 B5 B1051.6,15600


## Task 9: Failed drone landings in 2015

In [11]:
query="SELECT substr(Date,6,2) AS Month,Booster_Version,Launch_Site,Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome='Failure (drone ship)' AND substr(Date,1,4)='2015' ORDER BY Date"
print(query)
result=pd.read_sql_query(query,con)
answers['9']=result.to_dict('records')
display(result)

SELECT substr(Date,6,2) AS Month,Booster_Version,Launch_Site,Landing_Outcome FROM SPACEXTBL WHERE Landing_Outcome='Failure (drone ship)' AND substr(Date,1,4)='2015' ORDER BY Date


,Month,Booster_Version,Launch_Site,Landing_Outcome
0,01,F9 v1.1 B1012,CCAFS LC-40,Failure (drone ship)
1,04,F9 v1.1 B1015,CCAFS LC-40,Failure (drone ship)


## Task 10: Landing outcomes ranked 2010 to 2017

In [12]:
query="SELECT Landing_Outcome,COUNT(*) AS Outcomes FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Outcomes DESC,Landing_Outcome"
print(query)
result=pd.read_sql_query(query,con)
answers['10']=result.to_dict('records')
display(result)

SELECT Landing_Outcome,COUNT(*) AS Outcomes FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Outcomes DESC,Landing_Outcome


,Landing_Outcome,Outcomes
0,No attempt,10
1,Failure (drone ship),5
2,Success (drone ship),5
3,Controlled (ocean),3
4,Success (ground pad),3
5,Failure (parachute),2
6,Uncontrolled (ocean),2
7,Precluded (drone ship),1


## Save reproducible results

In [13]:
(OUT/'sql_results.json').write_text(json.dumps(answers,indent=2))

4119